In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

# ========================== LOAD + FIX SHAPE ==========================
print("Loading matrices...")
micro_basis  = np.load("Matrix_A_Micro_Basis.npy")
micro_labels = np.load("Matrix_A_Micro_Labels.npy")
macro_basis  = np.load("Matrix_A_Macro_Basis.npy")
macro_labels = np.load("Matrix_A_Macro_Labels.npy")

print(f"Micro shape: {micro_basis.shape}")
print(f"Macro shape: {macro_basis.shape}")

# Pad the smaller matrix so both have the same number of columns
if micro_basis.shape[1] > macro_basis.shape[1]:
    pad = micro_basis.shape[1] - macro_basis.shape[1]
    macro_basis = np.pad(macro_basis, ((0, 0), (0, pad)), mode='constant')
    print(f"✅ Padded macro to {macro_basis.shape}")
else:
    pad = macro_basis.shape[1] - micro_basis.shape[1]
    micro_basis = np.pad(micro_basis, ((0, 0), (0, pad)), mode='constant')
    print(f"✅ Padded micro to {micro_basis.shape}")

# Combine
basis  = np.vstack([micro_basis, macro_basis])
labels = np.concatenate([micro_labels, macro_labels])

print(f"\n✅ Final combined basis: {basis.shape[0]} materials × {basis.shape[1]} bins")
print(f"Total materials: {len(labels)}")

# ========================== PYTORCH SURROGATE ==========================
class ScannerSurrogate(nn.Module):
    def __init__(self, input_dim, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
            nn.Linear(hidden//2, hidden//4), nn.ReLU(),
            nn.Linear(hidden//4, input_dim)
        )
    def forward(self, x):
        return self.net(x)

X = torch.tensor(basis, dtype=torch.float32).to(device)

train_idx, val_idx = train_test_split(np.arange(len(X)), test_size=0.15, random_state=42)
train_loader = DataLoader(TensorDataset(X[train_idx], X[train_idx]), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(X[val_idx],   X[val_idx]),   batch_size=32, shuffle=False)

model = ScannerSurrogate(input_dim=basis.shape[1]).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

print("\nTraining surrogate on 960M-particle data...")
for epoch in range(300):
    model.train()
    train_loss = 0.0
    for bx, target in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(bx), target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    if (epoch + 1) % 50 == 0 or epoch == 299:
        model.eval()
        val_loss = sum(criterion(model(bx), target).item() for bx, target in val_loader)
        print(f"Epoch {epoch+1:3d} | Train MSE: {train_loss/len(train_loader):.2e} | Val MSE: {val_loss/len(val_loader):.2e}")

torch.save(model.state_dict(), "surrogate_960M.pth")
print("✅ Surrogate saved!")

# ========================== VALIDATION PLOTS ==========================
model.eval()
with torch.no_grad():
    recon = model(X[:8]).cpu().numpy()

plt.figure(figsize=(14, 8))
for i in range(8):
    plt.subplot(4, 2, i+1)
    plt.semilogy(basis[i], label="True OpenMC (960M)", alpha=0.8)
    plt.semilogy(recon[i], label="Surrogate", alpha=0.8)
    plt.title(labels[i])
    plt.legend()
plt.tight_layout()
plt.suptitle("Surrogate vs True High-Fidelity Spectra")
plt.show()

# ========================== PyMC BAYESIAN ==========================
def run_pymc_surrogate(observed_spectrum, n_samples=1000):
    print("\n🔬 Running PyMC + Surrogate Bayesian inference...")
    model_sur = ScannerSurrogate(basis.shape[1]).to(device)
    model_sur.load_state_dict(torch.load("surrogate_960M.pth", map_location=device))
    model_sur.eval()

    with pm.Model() as m:
        proportions = pm.Dirichlet("proportions", a=np.ones(len(labels)))
        def surrogate_forward(p):
            with torch.no_grad():
                p_t = torch.tensor(p, dtype=torch.float32, device=device).unsqueeze(0)
                return model_sur(p_t).cpu().numpy().flatten()
        predicted = pm.Deterministic("predicted", surrogate_forward(proportions))
        sigma = pm.HalfNormal("sigma", sigma=0.05)
        pm.Normal("observed", mu=predicted, sigma=sigma, observed=observed_spectrum)
        trace = pm.sample(n_samples, tune=300, target_accept=0.95, chains=2, progressbar=True)
    
    summary = az.summary(trace, var_names=["proportions"], hdi_prob=0.95)
    print("\n🎯 BAYESIAN COMPOSITION")
    print("="*60)
    for i, label in enumerate(labels):
        m = summary.loc[f"proportions[{i}]", "mean"] * 100
        lo = summary.loc[f"proportions[{i}]", "hdi_2.5%"] * 100
        hi = summary.loc[f"proportions[{i}]", "hdi_97.5%"] * 100
        if m > 0.5:
            print(f"{label:25} {m:6.1f}%  [{lo:5.1f}–{hi:5.1f}]%")

# Quick test
dummy_obs = np.random.rand(basis.shape[1]) * 0.01
run_pymc_surrogate(dummy_obs, n_samples=800)

print("\n🎉 All done! Replace dummy_obs with real scan data when ready.")

🚀 Using device: cuda
Loading matrices...
Micro shape: (19990, 23)
Macro shape: (21989, 20)
✅ Padded macro to (21989, 23)

✅ Final combined basis: 41979 materials × 23 bins
Total materials: 43

Training surrogate on 960M-particle data...
Epoch  50 | Train MSE: 1.94e-08 | Val MSE: 4.48e-08
Epoch 100 | Train MSE: 1.52e-08 | Val MSE: 1.76e-08
Epoch 150 | Train MSE: 1.58e-08 | Val MSE: 7.43e-08
